# CODE 3.3: Check the Preprocessing

> **Goal:** make sure every file of step 3 is correct, and that no data can leak between train, validation and test, before any model uses the files.
>
> - **Input:** `data/processed/crops/` (notebooks 3.1 and 3.2), the split files in `splits/`, and the original data in `data/raw/chewinggum/` (to compare pixels)
> - **Output:** no files. Only a list of checks. At the end you should see **ALL CHECKS PASSED**.
>
> This notebook checks the files of 3.1 and 3.2 from the outside: it opens them the way YOLO and PatchCore will, and compares them with the original data.

### What Is Checked

> 1. **Crops:** one crop per photo, all the same size, cut exactly from the original photo, with the gum inside. The window size really comes from the normal train photos.
> 2. **Cropped masks:** cut with the same window, the same numbers as the original mask, and no real defect cut off.
> 3. **Label files:** the right format, inside the crop, and they match the masks.
> 4. **Photo lists and dataset files:** the right photos, the right number of copies, and only train photos in the train lists. YOLO finds every picture and every label file.

## Import Libraries & Variables

### Import Libraries

> - `os`: works with folders and file paths.
> - `Counter`: counts how often each line appears in a list.
> - `numpy`: works with numbers and pixels.
> - `pandas`: works with tables.
> - `PIL` (Pillow): opens photos, crops and masks.
> - `scipy.ndimage`: finds the pieces in a mask.
> - `tqdm`: shows a progress bar while the code goes through many photos.

In [1]:
import os
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage
from tqdm import tqdm

### Variables

#### Folder Paths

> This code finds the main project folder. It starts in the folder of this notebook and goes up until it finds the `data` folder, so the paths work wherever the notebook is.
>
> - `RAW_DIR`, `GUM_DIR`: the original data, to compare the crops with.
> - `SPLITS_DIR`, `SAME_GUM_FILE`: the split files and the photos of the same gum (notebooks 1.2 and 1.3).
> - `CROP_DIR`: the folder with all files of step 3.
> - `CROP_TABLE_FILE`, `BOX_TABLE_FILE`: the tables of notebooks 3.1 and 3.2.
>
> If a needed file is missing, the code stops with a message that says what to do.

In [2]:
ROOT_DIR = os.getcwd()
while not os.path.isdir(os.path.join(ROOT_DIR, "data")) and ROOT_DIR != os.path.dirname(ROOT_DIR):
    ROOT_DIR = os.path.dirname(ROOT_DIR)

RAW_DIR = os.path.join(ROOT_DIR, "data", "raw")
GUM_DIR = os.path.join(RAW_DIR, "chewinggum")
SPLITS_DIR = os.path.join(ROOT_DIR, "splits")
SAME_GUM_FILE = os.path.join(SPLITS_DIR, "same_gum_photos.csv")
CROP_DIR = os.path.join(ROOT_DIR, "data", "processed", "crops")
CROP_TABLE_FILE = os.path.join(CROP_DIR, "crop_boxes.csv")
BOX_TABLE_FILE = os.path.join(CROP_DIR, "yolo_boxes.csv")

if not os.path.exists(os.path.join(GUM_DIR, "image_anno.csv")):
    raise FileNotFoundError("Data not found. Copy the 'chewinggum' folder into data/raw/ (see README).")
if not os.path.exists(BOX_TABLE_FILE):
    raise FileNotFoundError("data/processed/crops/yolo_boxes.csv not found. Run notebooks 3.1 and 3.2 first.")
print("Project folder found:", os.path.basename(ROOT_DIR))

Project folder found: visual-defect-inspection


#### Expected Values

> The settings of notebooks 3.1 and 3.2, written here again so the files can be checked against them:
>
> - `MARGIN`, `SIZE_STEP`: the window rule of notebook 3.1.
> - `SPECK_PIXELS`, `MIN_BOX_SIDE`, `DEFECT_SHARE`: the speck size, the smallest box side and the share of defect lines of notebook 3.2.
> - `SHARE_TOLERANCE`: the defect share of a train list may be up to 3 percentage points away from one third, because the number of copies is rounded to a whole number.
> - `N_PHOTOS`, `SPLITS`, `LABEL_STEPS`: 603 photos, 3 splits, and k = 5, 10, 20, 40 labeled defect photos.
> - `MASK_NUMBERS`: the numbers a mask may hold (0 = no defect, 1 to 5 = defect type).
> - `CLASS_NAMES`: the 5 YOLO classes, in the order of notebook 3.2.
> - `TOUCHING`: which pixels count as touching (also corner to corner), as in notebooks 2.1 to 3.2.

In [3]:
MARGIN = 64
SIZE_STEP = 32
SPECK_PIXELS = 10
MIN_BOX_SIDE = 16
DEFECT_SHARE = 1 / 3
SHARE_TOLERANCE = 3
N_PHOTOS = 603
SPLITS = [1, 2, 3]
LABEL_STEPS = [5, 10, 20, 40]
MASK_NUMBERS = {0, 1, 2, 3, 4, 5}
CLASS_NAMES = {0: "chunk of gum missing", 1: "corner missing", 2: "scratches",
               3: "similar colour spot", 4: "small cracks"}
TOUCHING = np.ones((3, 3), dtype=bool)

## Functions

### Record a Check

> `check()` prints PASS or FAIL for one check and stores the result in `results`, with the name of the check as the key. Running a cell again never counts a check twice.

In [4]:
results = {}

def check(name, ok):
    results[name] = bool(ok)
    print(("PASS  " if ok else "FAIL  ") + name)

### Small Helpers

> - `full_path()`: turns a path from a table (written with `/`) into a real path on this computer.
> - `round_up()`: the smallest multiple of `step` that is not smaller than `value` (the same rule as notebook 3.1).
> - `copies_for()`: the number of copies of each defect photo in a train list (the same formula as notebook 3.2).
> - `read_lines()`: reads a text file and returns its lines, without empty lines.

In [5]:
def full_path(base_dir, rel_path):
    return os.path.join(base_dir, *rel_path.split("/"))


def round_up(value, step):
    return int(np.ceil(value / step)) * step


def copies_for(k, n_normal, defect_share):
    return round(defect_share / (1 - defect_share) * n_normal / k)


def read_lines(path):
    with open(path, encoding="utf-8") as f:
        return [line for line in f.read().splitlines() if line.strip()]

### Read a Dataset File

> `read_dataset_file()` reads one of our `.yaml` dataset files. A YAML library is not installed yet (it comes with Ultralytics in step 4), and our files are simple, so we read them by hand:
>
> - a line `key: value` becomes an entry of the result, like `train` → `split_1_k5_train.txt`,
> - the lines under `names:` start with two spaces, like `  0: chunk of gum missing`, and become the class names.

In [6]:
def read_dataset_file(path):
    content = {"names": {}}
    for line in read_lines(path):
        key, value = line.split(":", 1)
        if line.startswith("  "):
            content["names"][int(key)] = value.strip()
        elif value.strip():
            content[key.strip()] = value.strip()
    return content

### YOLO's Path Rules

> These two functions copy the rules that YOLO (Ultralytics) uses to find the files, from its own source code (`get_img_files()` and `img2label_paths()`):
>
> - `yolo_image_path()`: a list line that starts with `./` gets the folder of the list file in front of it. Then all `/` become the separator of this computer (`os.sep`, which is `\` on Windows).
> - `yolo_label_path()`: the last `images` folder in the path becomes `labels`, and the file ending becomes `.txt`.
>
> If the files exist at the paths these rules give, YOLO will find them too.

In [7]:
def yolo_image_path(line, list_file):
    folder = os.path.dirname(list_file) + os.sep
    path = line.replace("./", folder, 1) if line.startswith("./") else line
    return path.replace("/", os.sep)


def yolo_label_path(image_path):
    images, labels = f"{os.sep}images{os.sep}", f"{os.sep}labels{os.sep}"
    return labels.join(image_path.rsplit(images, 1)).rsplit(".", 1)[0] + ".txt"

### Read a Label File

> `read_boxes()` reads a label file and turns every line back into a box in pixels of the crop: *left = (x_center − width ÷ 2) × crop width*, and so on, rounded to whole pixels. It returns the boxes and whether every line has the right format: 5 numbers, a whole class number from 0 to 4, and 4 numbers between 0 and 1.

In [8]:
def read_boxes(path, crop_width, crop_height):
    boxes, good_format = [], True
    for line in read_lines(path):
        parts = line.split()
        numbers = [float(p) for p in parts]
        good_format &= (len(parts) == 5 and numbers[0] in CLASS_NAMES and all(0 <= n <= 1 for n in numbers[1:]))
        if len(parts) == 5:
            c, x, y, w, h = numbers
            boxes.append({"class": int(c), "left": round((x - w / 2) * crop_width), "top": round((y - h / 2) * crop_height),
                          "right": round((x + w / 2) * crop_width), "bottom": round((y + h / 2) * crop_height)})
    return boxes, good_format

## Load Data

### Load the Tables

> - `photos`: the original photo list, with `is_defect`.
> - `crops`: the crop table of notebook 3.1.
> - `boxes`: the box table of notebook 3.2.
> - `splits`: the 3 split files.
> - `left_out`: the photos that notebook 1.2 left out (the second photo of a gum photographed twice).

In [9]:
photos = pd.read_csv(os.path.join(GUM_DIR, "image_anno.csv")).fillna("")
photos["is_defect"] = photos["label"] != "normal"
crops = pd.read_csv(CROP_TABLE_FILE).fillna({"crop_mask": ""})
boxes = pd.read_csv(BOX_TABLE_FILE)
splits = {s: pd.read_csv(os.path.join(SPLITS_DIR, f"split_{s}.csv")) for s in SPLITS}
same_gum = pd.read_csv(SAME_GUM_FILE)
left_out = set(same_gum.loc[~same_gum["keep"], "image"])

crops = crops.merge(photos[["image", "is_defect"]], on="image", how="left")
print(f"Photos: {len(photos)} | crop table rows: {len(crops)} | boxes: {len(boxes)} | left-out photos: {len(left_out)}")

Photos: 603 | crop table rows: 603 | boxes: 205 | left-out photos: 6


## Check the Crops

### The Crop Table and the Window Size

> - Every photo has exactly one row.
> - All windows have the same size, a multiple of `SIZE_STEP`.
> - The window size follows the rule of notebook 3.1, computed again here from the gum boxes in the table: *biggest gum of the normal train photos + 2 × MARGIN, rounded up*, for each split. This proves that only normal train photos decided it.
> - Every window lies inside its photo, holds its whole gum box, and has the gum box in its middle.

In [10]:
check(f"crop_boxes.csv has one row for each of the {N_PHOTOS} photos",
      len(crops) == N_PHOTOS and set(crops["image"]) == set(photos["image"]))
check("each photo appears only once in crop_boxes.csv", crops["image"].is_unique)

widths = crops["crop_right"] - crops["crop_left"]
heights = crops["crop_bottom"] - crops["crop_top"]
WINDOW = (int(widths.iloc[0]), int(heights.iloc[0]))
check(f"all windows have the same size ({WINDOW[0]} x {WINDOW[1]})", widths.nunique() == 1 and heights.nunique() == 1)
check(f"the window size is a multiple of {SIZE_STEP}", WINDOW[0] % SIZE_STEP == 0 and WINDOW[1] % SIZE_STEP == 0)

gum_width = crops["gum_right"] - crops["gum_left"]
gum_height = crops["gum_bottom"] - crops["gum_top"]
rule_sizes = []
for s, split in splits.items():
    normal_train = crops["image"].isin(split.loc[(split["split"] == "train") & ~split["is_defect"], "image"])
    rule_sizes.append((round_up(gum_width[normal_train].max() + 2 * MARGIN, SIZE_STEP),
                       round_up(gum_height[normal_train].max() + 2 * MARGIN, SIZE_STEP)))
print("Window size from the normal train photos of each split:", rule_sizes)
check("the window size is the biggest of the sizes the 3 splits give (rule of notebook 3.1)",
      WINDOW == (max(w for w, h in rule_sizes), max(h for w, h in rule_sizes)))

photo_sizes = [Image.open(os.path.join(RAW_DIR, p)).size for p in crops["image"]]
check("every window lies inside its photo",
      all(row.crop_left >= 0 and row.crop_top >= 0 and row.crop_right <= w and row.crop_bottom <= h
          for row, (w, h) in zip(crops.itertuples(), photo_sizes)))
check("every gum box lies inside its window",
      ((crops["gum_left"] >= crops["crop_left"]) & (crops["gum_top"] >= crops["crop_top"])
       & (crops["gum_right"] <= crops["crop_right"]) & (crops["gum_bottom"] <= crops["crop_bottom"])).all())
off_x = (crops["crop_left"] + crops["crop_right"]) / 2 - (crops["gum_left"] + crops["gum_right"]) / 2
off_y = (crops["crop_top"] + crops["crop_bottom"]) / 2 - (crops["gum_top"] + crops["gum_bottom"]) / 2
check("every window has its gum box in the middle (at most 1 pixel off)", (off_x.abs() <= 1).all() and (off_y.abs() <= 1).all())

PASS  crop_boxes.csv has one row for each of the 603 photos
PASS  each photo appears only once in crop_boxes.csv
PASS  all windows have the same size (928 x 608)
PASS  the window size is a multiple of 32
Window size from the normal train photos of each split: [(928, 608), (928, 608), (928, 608)]
PASS  the window size is the biggest of the sizes the 3 splits give (rule of notebook 3.1)
PASS  every window lies inside its photo
PASS  every gum box lies inside its window
PASS  every window has its gum box in the middle (at most 1 pixel off)


### The Pixels of Every Crop and Every Cropped Mask

> For every photo we cut the window out of the original photo again and compare it with the saved crop, pixel by pixel. The same for every mask. Because PNG keeps every pixel, they must be exactly equal.
>
> For the masks we also check the numbers they hold, that every defect photo still has defect pixels after cropping, and which defect pixels were cut off. A cut-off pixel is fine only if it belongs to a speck: a piece of its defect type with fewer than `SPECK_PIXELS` pixels, which notebook 3.2 drops anyway.

In [11]:
crop_ok, crop_same, mask_ok, mask_same, mask_numbers, mask_not_empty, cut_ok, cut_found = [], [], [], [], set(), [], [], []
for _, row in tqdm(crops.iterrows(), total=len(crops), desc="Comparing every crop with its original"):
    window = (row["crop_left"], row["crop_top"], row["crop_right"], row["crop_bottom"])
    crop = Image.open(full_path(CROP_DIR, row["crop_image"]))
    crop_ok.append(crop.size == WINDOW and crop.mode == "RGB")
    original = Image.open(os.path.join(RAW_DIR, row["image"])).convert("RGB").crop(window)
    crop_same.append(np.array_equal(np.asarray(crop), np.asarray(original)))
    if not row["crop_mask"]:
        continue
    cropped_mask = np.asarray(Image.open(full_path(CROP_DIR, row["crop_mask"])))
    raw_mask = np.asarray(Image.open(os.path.join(RAW_DIR, photos.loc[photos["image"] == row["image"], "mask"].iloc[0])))
    mask_ok.append(cropped_mask.shape == (WINDOW[1], WINDOW[0]))
    mask_same.append(np.array_equal(cropped_mask, raw_mask[window[1]:window[3], window[0]:window[2]]))
    mask_numbers |= set(np.unique(cropped_mask).tolist())
    mask_not_empty.append((cropped_mask > 0).any())
    inside = np.zeros(raw_mask.shape, dtype=bool)
    inside[window[1]:window[3], window[0]:window[2]] = True
    for number in set(np.unique(raw_mask[~inside]).tolist()) - {0}:
        pieces, n_pieces = ndimage.label(raw_mask == number, structure=TOUCHING)
        sizes = np.bincount(pieces.ravel())
        cut_ok.append(all(sizes[p] < SPECK_PIXELS for p in np.unique(pieces[(raw_mask == number) & ~inside])))
        cut_found.append(f"{row['image'].split('Images/')[-1]} ({CLASS_NAMES[number - 1]})")

check(f"every crop exists, is RGB and {WINDOW[0]} x {WINDOW[1]} pixels", all(crop_ok))
check("every crop has exactly the pixels of its window in the original photo", all(crop_same))
check("every defect photo has a cropped mask, and no normal photo has one",
      ((crops["crop_mask"] != "") == crops["is_defect"]).all())
check("every cropped mask has the size of its crop", len(mask_ok) == crops["is_defect"].sum() and all(mask_ok))
check("every cropped mask has exactly the numbers of its window in the original mask", all(mask_same))
check(f"cropped masks only hold the numbers 0 to 5 (found: {sorted(mask_numbers)})", mask_numbers <= MASK_NUMBERS)
check("every defect photo still has defect pixels after cropping", all(mask_not_empty))
print("Defect pixels outside a window found in:", cut_found)
check(f"every defect pixel outside a window belongs to a speck (fewer than {SPECK_PIXELS} pixels)", all(cut_ok))

Comparing every crop with its original:   0%|          | 0/603 [00:00<?, ?it/s]

Comparing every crop with its original: 100%|██████████| 603/603 [00:30<00:00, 19.59it/s]

PASS  every crop exists, is RGB and 928 x 608 pixels
PASS  every crop has exactly the pixels of its window in the original photo
PASS  every defect photo has a cropped mask, and no normal photo has one
PASS  every cropped mask has the size of its crop
PASS  every cropped mask has exactly the numbers of its window in the original mask
PASS  cropped masks only hold the numbers 0 to 5 (found: [0, 1, 2, 3, 4, 5])
PASS  every defect photo still has defect pixels after cropping
Defect pixels outside a window found in: ['Anomaly/080.JPG (small cracks)']
PASS  every defect pixel outside a window belongs to a speck (fewer than 10 pixels)


## Check the Label Files

### Format, Size and Agreement with the Masks

> - Every crop has a label file (at the path of YOLO's rule). Normal photos have an empty one, defect photos at least one line.
> - Every line has the right format, and every box lies inside the crop and is at least `MIN_BOX_SIDE` pixels wide and high.
> - The label files hold exactly the boxes of `yolo_boxes.csv`.
> - Every box holds pixels of its own defect type in the cropped mask (*class = mask number − 1*).
> - Every piece of `SPECK_PIXELS` or more pixels lies completely inside a box of its own type, so no real defect is left without a box.

In [12]:
label_exists, empty_ok, format_ok, inside_ok, size_ok, table_ok, type_ok, covered_ok = [], [], [], [], [], [], [], []
for _, row in tqdm(crops.iterrows(), total=len(crops), desc="Checking the label files"):
    path = yolo_label_path(full_path(CROP_DIR, row["crop_image"]))
    label_exists.append(os.path.exists(path))
    if not label_exists[-1]:
        continue
    found, good_format = read_boxes(path, WINDOW[0], WINDOW[1])
    format_ok.append(good_format)
    empty_ok.append((len(found) == 0) != row["is_defect"])
    inside_ok += [b["left"] >= 0 and b["top"] >= 0 and b["right"] <= WINDOW[0] and b["bottom"] <= WINDOW[1] for b in found]
    size_ok += [b["right"] - b["left"] >= MIN_BOX_SIDE and b["bottom"] - b["top"] >= MIN_BOX_SIDE for b in found]
    in_table = boxes.loc[boxes["image"] == row["image"], ["class", "left", "top", "right", "bottom"]]
    table_ok.append(sorted(map(tuple, in_table.to_numpy().tolist())) ==
                    sorted((b["class"], b["left"], b["top"], b["right"], b["bottom"]) for b in found))
    if not row["is_defect"]:
        continue
    mask = np.asarray(Image.open(full_path(CROP_DIR, row["crop_mask"])))
    type_ok += [(mask[b["top"]:b["bottom"], b["left"]:b["right"]] == b["class"] + 1).any() for b in found]
    for c in CLASS_NAMES:
        pieces, n_pieces = ndimage.label(mask == c + 1, structure=TOUCHING)
        sizes = np.bincount(pieces.ravel())
        for p in range(1, n_pieces + 1):
            if sizes[p] < SPECK_PIXELS:
                continue
            rows, cols = np.nonzero(pieces == p)
            covered_ok.append(any(b["class"] == c and b["left"] <= cols.min() and cols.max() < b["right"]
                                  and b["top"] <= rows.min() and rows.max() < b["bottom"] for b in found))

check(f"every crop has a label file ({sum(label_exists)} of {len(crops)})", all(label_exists))
check("normal photos have empty label files, defect photos have at least one box", all(empty_ok))
check("every label line has 5 numbers: a class from 0 to 4 and 4 numbers between 0 and 1", all(format_ok))
check(f"every box lies inside the crop ({len(inside_ok)} boxes)", all(inside_ok))
check(f"every box is at least {MIN_BOX_SIDE} pixels wide and high", all(size_ok))
check("the label files hold exactly the boxes of yolo_boxes.csv", all(table_ok) and len(inside_ok) == len(boxes))
check("every box holds pixels of its own defect type", all(type_ok))
check(f"every piece of {SPECK_PIXELS} or more pixels lies inside a box of its own type ({len(covered_ok)} pieces)",
      all(covered_ok))

Checking the label files: 100%|██████████| 603/603 [00:05<00:00, 106.65it/s]

PASS  every crop has a label file (603 of 603)
PASS  normal photos have empty label files, defect photos have at least one box
PASS  every label line has 5 numbers: a class from 0 to 4 and 4 numbers between 0 and 1
PASS  every box lies inside the crop (205 boxes)
PASS  every box is at least 16 pixels wide and high
PASS  the label files hold exactly the boxes of yolo_boxes.csv
PASS  every box holds pixels of its own defect type
PASS  every piece of 10 or more pixels lies inside a box of its own type (215 pieces)


## Check the Photo Lists and Dataset Files

### Every Split, Every k

> For every split:
>
> - **Validation and test lists:** exactly the validation (or test) photos of the split file, each once.
> - **Train lists (one per k):**
>   - every normal train photo exactly once: the same normal photos PatchCore learns from, which keeps the comparison fair,
>   - exactly the k defect photos of `yolo_k`, each `copies_for()` times,
>   - only train photos of this split (**no validation or test photo**: the most important leakage check),
>   - a defect share close to one third,
>   - the defect photos of the smaller k are all in it.
> - **Dataset files:** the right three lists of this split and this k, which exist next to the file; the 5 class names in the right order; and no `path` line.
> - **Leakage:** no photo is in two of the train, validation and test lists, and no left-out photo of notebook 1.2 is in any list.
>
> To know which photo a list line means, `photo_of` turns a line like `./images/Anomaly/005.png` back into the original photo path.

In [13]:
photo_of = {"./" + crop: image for image, crop in zip(crops["image"], crops["crop_image"])}
is_defect_photo = dict(zip(photos["image"], photos["is_defect"]))
all_list_files = []

for s, split in splits.items():
    print(f"\n=== split {s} ===")
    part_of = dict(zip(split["image"], split["split"]))
    lists = {}
    for part in ["val", "test"]:
        lists[part] = read_lines(os.path.join(CROP_DIR, f"split_{s}_{part}.txt"))
        all_list_files.append(os.path.join(CROP_DIR, f"split_{s}_{part}.txt"))
        wanted = set(split.loc[split["split"] == part, "image"])
        check(f"split {s}: the {part} list holds exactly the {len(wanted)} {part} photos, each once",
              len(lists[part]) == len(wanted) and {photo_of.get(line) for line in lists[part]} == wanted)

    normal_train = set(split.loc[(split["split"] == "train") & ~split["is_defect"], "image"])
    smaller_k_defects = set()
    for k in LABEL_STEPS:
        list_file = os.path.join(CROP_DIR, f"split_{s}_k{k}_train.txt")
        all_list_files.append(list_file)
        lines = read_lines(list_file)
        counts = Counter(photo_of.get(line) for line in lines)
        normal_counts = {p: n for p, n in counts.items() if not is_defect_photo.get(p, True)}
        defect_counts = {p: n for p, n in counts.items() if is_defect_photo.get(p, False)}
        copies = copies_for(k, len(normal_train), DEFECT_SHARE)
        yolo_defects = set(split.loc[split[f"yolo_{k}"] & split["is_defect"], "image"])
        share = 100 * sum(defect_counts.values()) / len(lines)

        check(f"split {s} k={k}: every one of the {len(normal_train)} normal train photos is in the train list exactly once",
              set(normal_counts) == normal_train and set(normal_counts.values()) == {1})
        check(f"split {s} k={k}: the train list holds the {k} YOLO defect photos, each {copies} times",
              set(defect_counts) == yolo_defects and set(defect_counts.values()) == {copies})
        check(f"split {s} k={k}: the train list only holds train photos of split {s}",
              all(part_of.get(p) == "train" for p in counts))
        check(f"split {s} k={k}: defect lines are {share:.1f}% of the train list (one third, give or take {SHARE_TOLERANCE})",
              abs(share - 100 * DEFECT_SHARE) <= SHARE_TOLERANCE)
        check(f"split {s} k={k}: the train list holds the defect photos of the smaller k", smaller_k_defects <= set(defect_counts))
        smaller_k_defects = set(defect_counts)
        lists[f"train_{k}"] = lines

        yaml_file = os.path.join(CROP_DIR, f"split_{s}_k{k}.yaml")
        content = read_dataset_file(yaml_file)
        wanted_files = {"train": f"split_{s}_k{k}_train.txt", "val": f"split_{s}_val.txt", "test": f"split_{s}_test.txt"}
        check(f"split {s} k={k}: the dataset file names the right lists, they exist, the 5 classes are in order, no 'path' line",
              all(content.get(key) == name and os.path.exists(os.path.join(CROP_DIR, name)) for key, name in wanted_files.items())
              and content["names"] == CLASS_NAMES and "path" not in content)

    train_photos = {photo_of.get(line) for k in LABEL_STEPS for line in lists[f"train_{k}"]}
    val_photos = {photo_of.get(line) for line in lists["val"]}
    test_photos = {photo_of.get(line) for line in lists["test"]}
    check(f"split {s}: no photo is in two of the train, val and test lists",
          not (train_photos & val_photos) and not (train_photos & test_photos) and not (val_photos & test_photos))
    check(f"split {s}: no left-out photo of notebook 1.2 is in any list", not (left_out & (train_photos | val_photos | test_photos)))


=== split 1 ===
PASS  split 1: the val list holds exactly the 60 val photos, each once
PASS  split 1: the test list holds exactly the 150 test photos, each once
PASS  split 1 k=5: every one of the 347 normal train photos is in the train list exactly once
PASS  split 1 k=5: the train list holds the 5 YOLO defect photos, each 35 times
PASS  split 1 k=5: the train list only holds train photos of split 1
PASS  split 1 k=5: defect lines are 33.5% of the train list (one third, give or take 3)
PASS  split 1 k=5: the train list holds the defect photos of the smaller k
PASS  split 1 k=5: the dataset file names the right lists, they exist, the 5 classes are in order, no 'path' line
PASS  split 1 k=10: every one of the 347 normal train photos is in the train list exactly once
PASS  split 1 k=10: the train list holds the 10 YOLO defect photos, each 17 times
PASS  split 1 k=10: the train list only holds train photos of split 1
PASS  split 1 k=10: defect lines are 32.9% of the train list (one third

### Can YOLO Find Every Picture and Label File?

> For every line of every list (`all_list_files`), we follow YOLO's path rules (`yolo_image_path()` and `yolo_label_path()`) and check that both files exist.

In [14]:
n_lines, missing = 0, []
for list_file in all_list_files:
    for line in read_lines(list_file):
        n_lines += 1
        image_path = yolo_image_path(line, list_file)
        if not (os.path.exists(image_path) and os.path.exists(yolo_label_path(image_path))):
            missing.append(line)
print(f"List files: {len(all_list_files)} | lines: {n_lines}")
check(f"every list line leads YOLO to an existing crop and label file ({len(missing)} missing)", not missing)

List files: 18 | lines: 6849
PASS  every list line leads YOLO to an existing crop and label file (0 missing)


## Result

### Final Result

> If every check passed, the files of step 3 are ready for step 4. If a check failed, its name is printed: run notebooks 3.1 and 3.2 again (Restart, then Run All), then this notebook.

In [15]:
failed = [name for name, ok in results.items() if not ok]
if results and not failed:
    print(f"ALL CHECKS PASSED ({len(results)} checks)")
else:
    print(f"{len(failed)} of {len(results)} checks FAILED:")
    for name in failed:
        print("  -", name)

ALL CHECKS PASSED (109 checks)


## Summary

> - **Crops:** one per photo, all 928 x 608 pixels, pixel-exact copies of their window, each with its gum inside and in the middle. The window size follows from the normal train photos only.
> - **Masks:** cut with the same windows, unchanged numbers, and only specks cut off.
> - **Labels:** right format, inside the crop, at least 16 pixels wide and high, in line with the masks, and every real defect piece is inside a box of its type.
> - **Lists:** every train list holds all normal train photos once and its k defect photos with the right number of copies, and **no validation, test or left-out photo**. YOLO's own path rules find every file.
>
> **Next:** step 4, training PatchCore and YOLO.